In [ ]:
# test from (1 ... 18)
n_images = {n_images_value}
# n_images = 10

# test (True, False)
# gwcs_to_fits_sip = {gwcs_to_fits_sip_value}
gwcs_to_fits_sip = True

# server size (small: 2 CPU + 16 GB RAM, medium: 8 CPU + 65 GB RAM)

In [ ]:
import requests
import zipfile
from pathlib import Path
from glob import glob

In [ ]:
# Check for mastDownload folder. If it doesn't exist, download and unzip from Box.
mastdownload_dir = Path('mastDownload')

if not mastdownload_dir.exists():
    print("Downloading Roman SCA data from Box...")
    
    # Box direct download URL
    box_url = "https://stsci.box.com/shared/static/03rpaw9n8moypd4c6hp2ityfgenfjrg6"
    output_filename = "mastDownload.zip"
    
    # Stream the download and write it out one chunk at a time
    with requests.get(box_url, stream=True) as response:
        response.raise_for_status()
        
        with open(output_filename, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)
    
    print(f"✓ Downloaded {output_filename}")
    
    # Unzip the file
    print("Extracting files...")
    with zipfile.ZipFile(output_filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    
    print("✓ Extraction complete")
    
    # Clean up the zip file
    Path(output_filename).unlink()
    print("✓ Cleaned up ZIP file")
else:
    print("✓ mastDownload folder found locally")

# Find the Roman SCA files
existing_downloads = glob(
    'mastDownload/roman/r0000101001001001001_0002*/*'
)

print(f"\nFound {len(existing_downloads)} Roman SCA files")
downloads = {'Local Path': [path for path in existing_downloads]}

In [ ]:
# Initialize Imviz:

import jdaviz as jd

jd.show('sidecar:split-right')

In [ ]:
# batch load the results

with jd.batch_load():
    for path in sorted(downloads['Local Path'])[:n_images]:
        jd.load(path, format='Image', gwcs_to_fits_sip=gwcs_to_fits_sip)

In [ ]:
# make out-of-bound regions transparent (this needs to be a separate cell for now)
for layer in jd.viewers['Image']._obj.glue_viewer.state.layers:
    layer.cmap_bad = (0, 0, 0, 0)

In [ ]:
# WCS link:
orientation = jd.plugins['Orientation']
orientation.align_by = 'WCS'

In [ ]:
# Effectively: press the Home button in the viewer toolbar
jd.viewers['Image']._obj.glue_viewer.reset_limits()